# Лабораторная работа №3

## Построение модели прогноза инфляции с сезонной компонентой

**Цель работы:** сформировать практические навыки анализа временных рядов, выделения сезонной компоненты, построения прогнозных моделей и оценки точности прогноза.

В этой работе вместо прогноза продаж используется согласованная с преподавателем предметная область: **прогнозирование месячных индексов инфляции в РФ** по категориям `food`, `non_food`, `services`, `total`.

## 1. Импорт библиотек и загрузка данных

Исходный набор содержит месячные индексы потребительских цен по Российской Федерации. Значение `idx` интерпретируется как индекс к предыдущему месяцу, а целевая переменная задана как `target = idx - 100`, то есть месячный темп инфляции в процентных пунктах.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from data.data_processing import prepare_data

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 50)

DATA_PATH = Path("data/inflation_long_format.csv")
SAVE_DIR = Path("models/save_models")

raw = pd.read_csv(DATA_PATH)
df = prepare_data(DATA_PATH)
metrics = pd.read_csv(SAVE_DIR / "model_metrics.csv")
predictions = pd.read_csv(SAVE_DIR / "predictions.csv")
seasonality = pd.read_csv(SAVE_DIR / "seasonality.csv")
future_forecast = pd.read_csv(SAVE_DIR / "future_forecast.csv")

raw.head()

## 2. Паспорт исходных данных

Ниже приведена краткая характеристика набора данных: период наблюдений, частота, количество строк, категории инфляции и наличие пропусков.

In [ ]:
passport = pd.DataFrame({
    "Параметр": [
        "Наименование показателя",
        "Источник данных",
        "Период данных (начало)",
        "Период данных (окончание)",
        "Количество наблюдений в raw",
        "Количество наблюдений после лагов",
        "Частота сбора данных",
        "Категории",
        "Наличие пропусков",
        "Целевая переменная"
    ],
    "Значение": [
        "Индексы потребительских цен / месячная инфляция",
        "Файл data/inflation_long_format.csv",
        raw["date"].min(),
        raw["date"].max(),
        len(raw),
        len(df),
        "Ежемесячно",
        ", ".join(sorted(raw["inflation_type"].unique())),
        f"{int(raw.isna().sum().sum())} пропусков",
        "target = idx - 100"
    ]
})
passport

## 3. Предварительный анализ временного ряда

Для визуальной оценки тренда и сезонности построим графики месячной инфляции по категориям. Ранние годы имеют экстремальные значения, поэтому современный период дополнительно рассматривается отдельно.

![График временного ряда](report_assets/time_series.png)

In [ ]:
plot_df = df.copy()
type_cols = [c for c in plot_df.columns if c.startswith("type_")]
plot_df["inflation_type"] = plot_df[type_cols].idxmax(axis=1).str.replace("type_", "", regex=False)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=False)

for inflation_type, part in plot_df.groupby("inflation_type"):
    axes[0].plot(part["date"], part["target"], label=inflation_type, linewidth=1.2)
axes[0].set_title("Месячная инфляция по категориям, весь период")
axes[0].set_ylabel("target = idx - 100")
axes[0].legend(ncol=4)

recent = plot_df[plot_df["date"] >= "2016-01-01"]
for inflation_type, part in recent.groupby("inflation_type"):
    axes[1].plot(part["date"], part["target"], label=inflation_type, linewidth=1.4)
axes[1].set_title("Месячная инфляция по категориям, современный период")
axes[1].set_ylabel("target = idx - 100")
axes[1].legend(ncol=4)

plt.tight_layout()
plt.show()

**Визуальная интерпретация.** В начале 1990-х наблюдаются сильные выбросы. Поэтому обычная линейная регрессия и Ridge чувствительны к масштабу и дают нестабильное качество. В современном периоде месячная инфляция имеет небольшой разброс, из-за чего `R2` может быть отрицательным даже при небольшой абсолютной ошибке. Поэтому в работе основными метриками считаются `MAE`, `MSE` и `MAPE` по индексу.

## 4. Подготовка данных

В обработке данных выполнены следующие шаги:

- дата приведена к типу `datetime`;
- месяц закодирован one-hot признаками `m_01` ... `m_12`;
- категория инфляции закодирована one-hot признаками `type_*`;
- рассчитаны лаги `lag_1`, `lag_2`, `lag_3`, `lag_6`, `lag_12`;
- рассчитаны скользящие средние `roll_3`, `roll_6`, `roll_12`;
- целевая переменная задана как `target = idx - 100`.

Разбиение выполнено по времени: обучение до 2019 года, validation за 2019-2022, test за 2023-2024, holdout за 2025 год.

In [ ]:
split_table = pd.DataFrame({
    "Выборка": ["train", "validation", "test", "holdout", "recent actual"],
    "Период": ["1992-2018", "2019-2022", "2023-2024", "2025", "2026, факт доступен частично"],
    "Назначение": [
        "обучение моделей",
        "подбор лучшей модели",
        "финальная проверка качества",
        "проверка устойчивости на новом годе",
        "контроль последних фактических наблюдений"
    ],
    "Количество строк": [
        len(df[df["date"] < "2019-01-01"]),
        len(df[(df["date"] >= "2019-01-01") & (df["date"] < "2023-01-01")]),
        len(df[(df["date"] >= "2023-01-01") & (df["date"] < "2025-01-01")]),
        len(df[(df["date"] >= "2025-01-01") & (df["date"] < "2026-01-01")]),
        len(df[df["date"] >= "2026-01-01"])
    ]
})
split_table

## 5. Расчет коэффициентов сезонности

В лабораторной работе требуется реализовать два подхода:

1. **Метод средних по годам:** значение месяца делится на среднемесячное значение соответствующего года, затем коэффициенты усредняются по одинаковым месяцам.
2. **Метод сводной таблицы:** среднее значение месяца делится на общее среднее значение ряда.

Ниже показаны оба варианта коэффициентов сезонности для общей инфляции.

![Сезонный профиль](report_assets/seasonality.png)

In [ ]:
seasonality_total = seasonality[seasonality["inflation_type"] == "total"].copy()
seasonality_total[[
    "month_num",
    "seasonality_by_year_method",
    "seasonality_pivot_method"
]]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(
    seasonality_total["month_num"],
    seasonality_total["seasonality_by_year_method"],
    marker="o",
    label="Метод средних по годам"
)
ax.plot(
    seasonality_total["month_num"],
    seasonality_total["seasonality_pivot_method"],
    marker="s",
    label="Метод сводной таблицы"
)
ax.axhline(1, color="black", linewidth=1, linestyle="--")
ax.set_title("Сезонный профиль общей инфляции")
ax.set_xlabel("Месяц")
ax.set_ylabel("Коэффициент сезонности")
ax.set_xticks(range(1, 13))
ax.legend()
plt.show()

**Интерпретация.** Коэффициент выше 1 означает, что в этом месяце индекс инфляции в среднем выше общего уровня. Коэффициент ниже 1 означает относительно более спокойный месяц. Для управленческих решений это аналогично сезонному профилю спроса: месяцы с повышенным коэффициентом требуют большего внимания при планировании бюджета, цен и закупок.

## 6. Построение прогнозных моделей

Вместо Excel-модели Хольта-Винтерса работа выполнена на Python с ML-моделями. Для сохранения логики лабораторной работы добавлены сезонные baseline-модели:

- `naive_lag_1`: прогноз равен прошлому месяцу;
- `naive_lag_12`: прогноз равен тому же месяцу прошлого года;
- `rolling_3`, `rolling_12`: прогноз на основе скользящих средних.

ML-модели:

- Linear Regression;
- Ridge Regression;
- Huber Regressor;
- Gradient Boosting Regressor.

Для Ridge, Huber и Gradient Boosting используется `GridSearchCV` с `TimeSeriesSplit`, чтобы не нарушать временную структуру ряда.

In [ ]:
val_metrics = metrics[metrics["dataset"] == "val_2019_2022"].sort_values("mae")
test_metrics = metrics[metrics["dataset"] == "test_2023_2024"].sort_values("mae")
holdout_metrics = metrics[metrics["dataset"] == "holdout_2025"].sort_values("mae")

val_metrics

 ## 7. Оценка точности моделей

Для оценки качества используются:

- **MAE**: средняя абсолютная ошибка в процентных пунктах инфляции;
- **MSE/RMSE**: среднеквадратичная ошибка и ее корень;
- **MAPE по индексу**: процентная ошибка относительно исходного индекса `idx`;
- **R2**: дополнительная метрика, но не основная, так как при малой дисперсии современной инфляции она может быть отрицательной.

![Сравнение MAE](report_assets/test_mae.png)

In [ ]:
compare = pd.concat([
    val_metrics.assign(period="validation"),
    test_metrics.assign(period="test"),
    holdout_metrics.assign(period="holdout")
])

pivot_mae = compare.pivot_table(index="model", columns="period", values="mae")
pivot_mae.sort_values("test")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_metrics = test_metrics.sort_values("mae")
ax.bar(plot_metrics["model"], plot_metrics["mae"])
ax.set_title("Сравнение моделей на test 2023-2024 по MAE")
ax.set_ylabel("MAE")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

**Вывод по качеству.** На validation и test лучший результат показывает `gradient_boosting`. При этом на holdout 2025 наиболее устойчивым оказался `huber`, что говорит о важности робастных моделей для временных рядов с выбросами. Простые сезонные baseline-модели остаются сильными ориентирами: если ML-модель не превосходит `lag_12` или `lag_1`, ее нельзя считать практически полезной.

## 8. Сопоставление факта и прогноза

Построим график фактических и прогнозных значений для лучшей модели на периодах test и holdout.

![Факт и прогноз](report_assets/actual_vs_pred.png)

In [ ]:
features = json.load(open(SAVE_DIR / "features.json", encoding="utf-8"))
best_model = features["best_model"]

best_preds = predictions[
    (predictions["model"] == best_model) &
    (predictions["inflation_type"] == "total") &
    (predictions["dataset"].isin(["test_2023_2024", "holdout_2025", "recent_actual"]))
].copy()
best_preds["date"] = pd.to_datetime(best_preds["date"])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(best_preds["date"], best_preds["true_target"], marker="o", label="Факт")
ax.plot(best_preds["date"], best_preds["pred_target"], marker="s", label=f"Прогноз: {best_model}")
ax.set_title("Факт и прогноз месячной инфляции, категория total")
ax.set_ylabel("target = idx - 100")
ax.legend()
plt.show()

## 9. Итоговый прогноз на 12 месяцев

После оценки качества модель переобучается на всех доступных наблюдениях и строит итеративный прогноз на 12 месяцев вперед. Для каждого будущего месяца лаги и скользящие средние пересчитываются с учетом уже полученных прогнозных значений.

![Итоговый прогноз](report_assets/future_forecast.png)

In [ ]:
future_forecast["date"] = pd.to_datetime(future_forecast["date"])
forecast_total = future_forecast[future_forecast["inflation_type"] == "total"].copy()
forecast_total[["date", "inflation_type", "pred_target", "pred_idx"]]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(forecast_total["date"], forecast_total["pred_target"], marker="o")
ax.set_title("График итогового прогноза общей инфляции на 12 месяцев")
ax.set_xlabel("Дата")
ax.set_ylabel("Прогноз target = idx - 100")
plt.show()

## 10. Бизнес-выводы

1. У инфляционного ряда есть выраженные выбросы в начале периода, поэтому робастные модели важнее обычной линейной регрессии.
2. В современном периоде инфляция имеет малый разброс, поэтому MAE и MAPE информативнее, чем R2.
3. Сезонные лаги являются сильным benchmark: модель должна превосходить `lag_1` и `lag_12`, иначе ее практическая ценность сомнительна.
4. На test 2023-2024 лучший результат показал Gradient Boosting, а на holdout 2025 — Huber Regressor. Это означает, что для финального промышленного прогноза можно использовать либо Huber как более устойчивый вариант, либо ансамбль Huber + Gradient Boosting.
5. Для управленческих решений прогноз инфляции можно использовать при планировании индексации цен, бюджетировании закупок, оценке будущих расходов и выборе месяцев для корректировки ценовой политики.

## 11. Ответы на контрольные вопросы

**1. Компоненты временного ряда.** Временной ряд состоит из тренда, сезонной компоненты и случайной компоненты. Тренд отражает долгосрочное направление изменения, сезонность — регулярные повторения по месяцам или кварталам, случайная компонента — нерегулярные отклонения.

**2. Коэффициент сезонности.** Коэффициент сезонности показывает, насколько конкретный месяц выше или ниже среднего уровня. Например, коэффициент 1.25 для декабря означает, что значение в декабре в среднем на 25% выше обычного уровня.

**3. Аддитивная и мультипликативная модели.** В аддитивной модели сезонное отклонение прибавляется к тренду, в мультипликативной — умножается на уровень ряда. Если амплитуда сезонности растет вместе с уровнем ряда, уместнее мультипликативная модель.

**4. MAPE.** MAPE показывает среднюю абсолютную процентную ошибку прогноза. Обычно MAPE ниже 10% считается высокой точностью, 10-20% — хорошей, 20-50% — удовлетворительной. Но при значениях ряда около нуля MAPE может вводить в заблуждение.

**5. Использование прогноза.** Прогноз можно применять в бюджетировании, закупках, управлении ценами и финансовом планировании. Например, ожидаемый рост инфляции может быть основанием для пересмотра цен, корректировки запасов и обновления планов расходов.